# 02 · ShopDesk: from chat traces to SFT to RL

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daseinlabs/open-jev/blob/main/cookbooks/02-shopdesk-traces-to-rl/cookbook.ipynb)


The whole lifecycle on one realistic problem. You take the logs of an existing support agent, turn
them into a decision dataset, fine-tune a small model on them, measure it with deferral, and then
use RL to fix a mistake the logs can't teach. About 45–60 minutes, mostly Gemma reading text.

```
1  traces        an imperfect agent's chat logs (OpenAI format, with outcomes)
2  dataset       an adapter cuts each conversation into decisions
3  SFT           train, calibrate, test
4  play          the model runs fresh conversations, with and without deferral
5  RL            improve: learn from reward what the traces never showed
```


**Runs on** an NVIDIA/AMD GPU with about 9 GB of memory (PyTorch) or an Apple silicon Mac (MLX); the
backend is picked for you. On Colab, choose an L4 or A100 runtime: a T4 has no bfloat16, and
Gemma in float32 needs about 17 GB. The backbone (Gemma 3 4B, 8 GB) downloads once from
[`parsecai/gemma-3-4b-it`](https://huggingface.co/parsecai/gemma-3-4b-it). Accept the Gemma terms on that page first.

In [1]:
# Setup: install the SDK if it's missing, log in to Hugging Face once (the backbone is gated).
import importlib.util
if importlib.util.find_spec('openjev') is None:
    %pip install -q openjev-sdk

from huggingface_hub import get_token, login
if get_token() is None:
    login()

import json, shutil
from pathlib import Path
from openjev import compute

def fresh(path):
    """Remove a previous run's output, so every cell can be re-run."""
    shutil.rmtree(path, ignore_errors=True)
    return Path(path)

print('backend:', compute.backend(), '| device:', 'metal' if compute.backend() == 'mlx' else compute.device())

backend: mlx | device: metal


## The problem

ShopDesk (`openjev.tod.envs.shopdesk`) is a simulated retail support desk. A customer writes in
about one of their orders, and the agent calls tools until the request is resolved:

| tool | when it's right |
|---|---|
| `get_order(order_id)` | first: the status is only visible after a lookup |
| `cancel_order(order_id)` | cancel or return, order still pending |
| `return_order(order_id)` | cancel or return, delivered, return window **open** |
| `transfer_to_human()` | shipped (can't cancel in transit), or return window **closed** |
| `send_tracking(order_id)` | the customer asked where the order is |

The traces come from a **teacher** agent that is right most of the time, but sometimes skips the
lookup, picks the wrong order, or gives up. It also has a **systematic blind spot**: it ignores the
return window. Every conversation where that matters fails, so no successful trace shows what to do
with a closed window. RL closes that gap in step 5.

## Step 1 · Traces

In production, export your agent's logs: OpenAI-style messages with tool calls, plus an outcome per
conversation. Here, the teacher writes 1,500 of them.

In [2]:
from openjev.tod.envs.shopdesk import record_trace

RUN = Path('runs/shopdesk')
RUN.mkdir(parents=True, exist_ok=True)
traces = [record_trace(f'conv-{i:05d}', seed=i * 2) for i in range(1500)]
with open(RUN/'traces.jsonl', 'w') as stream:
    stream.writelines(json.dumps(t) + '\n' for t in traces)
print(f"{len(traces)} traces, teacher success {sum(t['outcome'] for t in traces) / len(traces):.1%}\n")
for m in traces[0]['messages']:
    print(f"{m['role']:9}", m.get('content') or json.dumps([c['function'] for c in m['tool_calls']]))

1500 traces, teacher success 72.1%

system    You are ShopDesk, a retail support agent. Look orders up before acting on them.
Customer profile: {"name": "Ivan", "orders": [{"id": "W9376", "item": "phone case"}, {"id": "W7634", "item": "blue kettle"}, {"id": "W8808", "item": "coffee grinder"}]}
user      I need to cancel my coffee grinder order.
assistant [{"name": "cancel_order", "arguments": "{\"order_id\": \"W8808\"}"}]
tool      {"ok": true}
assistant Thanks, is there anything else I can help with?


## Step 2 · Traces → decisions (the adapter)

This is the part you write for your own data, and the part to read most closely.

- **Two kinds:** `action` (which tool next) and `arg.order_id` (which order the call is about).
- **Cutting a conversation:** each tool call yields one `action` row and, if the tool takes an
  order, one `arg.order_id` row. The state is the conversation *before* the call; the label is
  what the agent actually did (`taken=`).
- **`render_state` is shared with the live environment**, so the model trains on exactly the view
  it gets when serving. Most train/serve bugs come from these two drifting apart.

In [3]:
from openjev.tod import Candidate, DecisionSpec, Episode, Kind, build_dataset
from openjev.tod.envs.shopdesk import TAKES_ORDER, TOOLS, render_state

ACTIONS = [Candidate(name, description, payload=name) for name, description in TOOLS.items()]

def order_candidates(state):
    return [Candidate(o['id'], f"Order {o['id']}: {o['item']}", payload=o['id']) for o in state['orders']]

SPEC = DecisionSpec('shopdesk', [
    Kind('action', 'Which tool should the agent call next?', lambda state: ACTIONS, version='1'),
    Kind('arg.order_id', 'Which order should this tool call use?', order_candidates, version='1'),
])

def trace_to_episode(trace):
    """One conversation -> decision rows, labelled with what the agent actually did."""
    messages = trace['messages']
    customer = json.loads(messages[0]['content'].split('Customer profile: ', 1)[1])
    message = next(m['content'] for m in messages if m['role'] == 'user')
    results = {m['tool_call_id']: json.loads(m['content']) for m in messages if m['role'] == 'tool'}
    history, decisions = [], []
    for m in messages:
        for call in m.get('tool_calls') or []:
            tool = call['function']['name']
            order_id = json.loads(call['function']['arguments']).get('order_id')
            row = f"{trace['id']}:{len(history)}"
            decisions.append(SPEC.decision(f'{row}:action', 'action',
                                           render_state(customer, message, history), taken=tool))
            if tool in TAKES_ORDER:
                decisions.append(SPEC.decision(f'{row}:order', 'arg.order_id',
                                               render_state(customer, message, history, next_tool=tool),
                                               taken=order_id))
            history.append((tool, order_id, results[call['id']]))
    return Episode(trace['id'], tuple(decisions), outcome=trace['outcome'])

for d in trace_to_episode(traces[0]).decisions[:2]:
    print(d.kind, '|', d.question, '\n  state:', json.dumps(d.state)[:300],
          '\n  candidates:', [c.id for c in d.candidates], '\n  label:', d.target, '\n')

action | Which tool should the agent call next? 
  state: {"customer_message": "I need to cancel my coffee grinder order.", "orders": [{"id": "W9376", "item": "phone case"}, {"id": "W7634", "item": "blue kettle"}, {"id": "W8808", "item": "coffee grinder"}], "tool_calls": []} 
  candidates: ['get_order', 'cancel_order', 'return_order', 'send_tracking', 'transfer_to_human'] 
  label: Choice(choice_id='cancel_order') 

arg.order_id | Which order should this tool call use? 
  state: {"customer_message": "I need to cancel my coffee grinder order.", "orders": [{"id": "W9376", "item": "phone case"}, {"id": "W7634", "item": "blue kettle"}, {"id": "W8808", "item": "coffee grinder"}], "tool_calls": [], "next_tool": "cancel_order"} 
  candidates: ['W9376', 'W7634', 'W8808'] 
  label: Choice(choice_id='W8808') 



`min_outcome=1.0` keeps only successful conversations, so the model doesn't learn the teacher's
random mistakes. `recall_miss_rate` is 0 here: every tool and order the agent used was among the
candidates. On real data, check it first.

In [4]:
dataset = build_dataset(SPEC, [trace_to_episode(t) for t in traces], min_outcome=1.0)
DATASET = fresh(RUN/'dataset')
dataset.write(DATASET)
print(json.dumps({k: dataset.report[k] for k in ('splits', 'recall_miss_rate', 'duplicates_dropped')}, indent=1))

{
 "splits": {
  "train": {
   "action": {
    "Choice": 1408
   },
   "arg.order_id": {
    "Choice": 1224
   }
  },
  "validation": {
   "action": {
    "Choice": 205
   },
   "arg.order_id": {
    "Choice": 182
   }
  },
  "calibration": {
   "action": {
    "Choice": 204
   },
   "arg.order_id": {
    "Choice": 180
   }
  },
  "test": {
   "action": {
    "Choice": 211
   },
   "arg.order_id": {
    "Choice": 175
   }
  }
 },
 "recall_miss_rate": {
  "action": 0.0,
  "arg.order_id": 0.0
 },
 "duplicates_dropped": 0
}


## Step 3 · Supervised fine-tuning

One head per kind, calibrated, measured on a held-out test split.

In [5]:
from openjev.tod.train import train_dataset

SFT = fresh(RUN/'sft')
sft = train_dataset(DATASET, None, SFT)

action: features (train)


Extracted 100/1408


Extracted 200/1408


Extracted 300/1408


Extracted 400/1408


...

{"epoch": 17, "train_loss": 0.06325315917823829, "val_top1": 0.9065934065934066, "val_top3": 1.0, "val_ece": 0.04021155776282236, "val_nll": 0.21842602102156675, "val_brier": 0.13133511775615597, "seconds": 0.2}


{"epoch": 18, "train_loss": 0.05229188993670582, "val_top1": 0.9120879120879121, "val_top3": 1.0, "val_ece": 0.026338347617066743, "val_nll": 0.2232167175638066, "val_brier": 0.12838878788850744, "seconds": 0.2}


{"epoch": 19, "train_loss": 0.055254570506756605, "val_top1": 0.8846153846153846, "val_top3": 1.0, "val_ece": 0.053842275399355835, "val_nll": 0.2831749486631387, "val_brier": 0.1609137361477885, "seconds": 0.2}


{"epoch": 20, "train_loss": 0.04624847586881491, "val_top1": 0.8956043956043956, "val_top3": 1.0, "val_ece": 0.0412418574691323, "val_nll": 0.27305935103441026, "val_brier": 0.1339026955132816, "seconds": 0.2}


calibrating and evaluating


In [6]:
for kind, r in sft['test'].items():
    print(f"{kind:13} accuracy {r['accuracy']:.1%}   mean confidence {r['mean_confidence']:.2f}   n={r['n']}")

action        accuracy 90.0%   mean confidence 0.90   n=211
arg.order_id  accuracy 90.3%   mean confidence 0.87   n=175


## Step 4 · Play fresh conversations, with deferral

Now the model *is* the agent. `HeadPolicy.rollout` asks the environment for the next decision,
builds it through `SPEC`, and executes the chosen payload. With a threshold, low-confidence
decisions go to the parent: an oracle here, your big model in production.

The 200 evaluation conversations use seeds never used for traces or training. `closed_window`
counts the blind-spot cases the model solves.

In [7]:
import random
from openjev.tod.envs.shopdesk import ShopDesk, teacher
from openjev.tod.rl import HeadPolicy

SEEDS = range(10_000_000, 10_000_200)

def teacher_success(seed):
    desk = ShopDesk().reset(seed)
    for tool, order_id in teacher(desk, random.Random(seed + 1)):
        desk.call(tool, order_id)
    return desk.success

def closed_window(policy):
    """Solved / total among conversations where the target's return window is closed."""
    solved = total = 0
    for seed in SEEDS:
        desk = ShopDesk()
        run = policy.rollout(desk, seed)
        t = desk.target
        if t['status'] == 'delivered' and t.get('return_window') == 'closed' and desk.intent != 'track':
            total += 1
            solved += run.reward == 1.0
    return f'{solved}/{total}'

def play(policy):
    report = {'closed_window': closed_window(policy)}
    for threshold in (None, 0.7, 0.9):
        r = policy.evaluate(ShopDesk, SEEDS, threshold=threshold)
        report[f'defer below {threshold}' if threshold else 'never defers'] = (
            f"success {r['mean_reward']:.1%}, decisions handled {r['decisions_handled']:.0%}")
    return report

print(f'teacher success {sum(map(teacher_success, SEEDS)) / len(SEEDS):.1%}')
sft_report = play(HeadPolicy(SFT, SPEC))
print(json.dumps(sft_report, indent=1))

teacher success 72.5%


{
 "closed_window": "9/24",
 "never defers": "success 75.0%, decisions handled 100%",
 "defer below 0.7": "success 96.5%, decisions handled 84%",
 "defer below 0.9": "success 100.0%, decisions handled 62%"
}


On its own, the SFT model is about as good as the agent it learned from. That's expected: it
imitates. Deferral is what makes it usable.

Almost all closed-window conversations fail. Every closed-window case in the traces was a
failure and was filtered out, so the model has never seen the right move. More data of the same
kind won't fix this.

## Step 5 · RL: learn what the traces never showed

`improve` plays 128 conversations per iteration. For each failure, it goes back through the
decisions **from the last one backwards**: it resets the desk to the same seed, replays up to that
decision, tries every other candidate, and lets the model finish. The latest decision where an
alternative succeeds becomes a training label. Labels are mixed with the SFT rows (the *anchor*),
so what the traces taught is kept. A kind whose validation accuracy drops more than 2 points is
rolled back (`rolled_back`).

Why this design:

1. **Try every alternative instead of sampling.** The SFT model gives `transfer_to_human` ~0%
   probability in closed-window states, so plain policy gradient (`reinforce`) never tries it.
2. **Label only the latest decision.** An early alternative can win on information the model
   didn't have yet. "Transfer at the very first turn" wins whenever the hidden window is closed.
3. **Roll back regressions.** Some failures can't be fitted without forgetting others.
4. **Weigh every row the same.** Each update averages over up to 32 labels and 256 anchor rows
   together. If the few labels were averaged as their own term, each would count as much as eight
   anchor rows, and the head would learn "return request → transfer" instead of "closed window → transfer".

In [8]:
from openjev.tod.rl import improve
from openjev.tod.train import evaluate_dataset

RL = fresh(RUN/'rl')
improve(HeadPolicy(SFT, SPEC), ShopDesk, RL, max_reward=1.0)
rl_test = evaluate_dataset(RL, DATASET, refit=True)  # the head changed, so refit calibration
for kind, r in rl_test['test'].items():
    print(f"{kind:13} accuracy {r['accuracy']:.1%} (SFT {sft['test'][kind]['accuracy']:.1%})")

{"iteration": 1, "greedy_mean_reward": 0.796875, "new_labels": 26, "total_labels": 26, "updated": ["action"], "rolled_back": ["arg.order_id"], "seconds": 74.0}


{"iteration": 2, "greedy_mean_reward": 0.8515625, "new_labels": 18, "total_labels": 44, "updated": ["action"], "rolled_back": ["arg.order_id"], "seconds": 69.9}


{"iteration": 3, "greedy_mean_reward": 0.7734375, "new_labels": 29, "total_labels": 73, "updated": ["action", "arg.order_id"], "rolled_back": [], "seconds": 71.6}


action        accuracy 90.0% (SFT 90.0%)
arg.order_id  accuracy 84.6% (SFT 90.3%)


In [9]:
rl_report = play(HeadPolicy(RL, SPEC))
print(f"{'':16} {'SFT':40} SFT + RL")
for key in sft_report:
    print(f'{key:16} {sft_report[key]:40} {rl_report[key]}')

                 SFT                                      SFT + RL
closed_window    9/24                                     24/24
never defers     success 75.0%, decisions handled 100%    success 82.5%, decisions handled 100%
defer below 0.7  success 96.5%, decisions handled 84%     success 97.0%, decisions handled 83%
defer below 0.9  success 100.0%, decisions handled 62%    success 100.0%, decisions handled 64%


RL fixes the blind spot (`closed_window`) and lifts success without deferral. Tool-choice test
accuracy holds; order accuracy can lose a few points, because the order head's RL labels are hard
cases. Read `rolled_back` above and the test report, and set each kind's deferral threshold from
the refitted calibration.

## What carries over to your data

- **Write the adapter for your logs**, or let [cookbook 03](../03-your-own-chat-traces/cookbook.ipynb)
  derive it from your tool schemas.
- **SFT copies your agent.** Filtering by outcome removes random mistakes, not systematic ones.
- **RL needs an `Environment`**: a simulator, a sandbox copy of your system, or a replay harness that
  can score outcomes. It needs only `reset`, `observe`, `act`, `defer` and `reward`.